# Atomic Scraper Service - Full API Test

This notebook tests all API endpoints of the service.

**Prerequisites:**
- Docker with `docker compose up -d` (Redis + services)
- OR run individually:
  - Redis must be running
  - Taskiq worker: `uv run taskiq worker src.infrastructure.queue.broker:broker`
  - FastAPI server: `uv run uvicorn src.api.main:app --reload`
- Playwright installed: `uv run playwright install --with-deps chromium`

**Base URL:** `http://localhost:8000`
**API Key:** `default_internal_key`

In [1]:
import requests
import json
import time
import os

In [2]:
def show_response(response, label=None, max_len=3000):
    """Helper function for JSON output with truncation"""
    if label:
        print(f"\n{'='*60}")
        print(f"{label}")
        print('='*60)
    print(f"Status: {response.status_code}")
    try:
        data = response.json()
        text = json.dumps(data, indent=2, ensure_ascii=False)
        if len(text) > max_len:
            print(text[:max_len] + f"\n... [{len(text) - max_len} more chars]")
        else:
            print(text)
    except Exception as e:
        print(f"(Failed to parse JSON: {e})")
        print(response.text[:max_len] if response.text else "(empty response)")

def assert_status(response, expected, label=None):
    """Assert status code and print result"""
    actual = response.status_code
    if label:
        print(f"{label}: ", end="")
    if actual == expected:
        print(f"✓ PASS ({actual})")
        return True
    else:
        print(f"✗ FAIL (expected {expected}, got {actual})")
        show_response(response)
        return False

def assert_json_key(response, key, label=None):
    """Assert response JSON contains a specific key"""
    try:
        data = response.json()
        has_key = key in data
        if label:
            print(f"{label}: ", end="")
        if has_key:
            print(f"✓ PASS (key '{key}' present)")
            return True
        else:
            print(f"✗ FAIL (key '{key}' NOT found)")
            return False
    except Exception as e:
        print(f"✗ FAIL (JSON parse error: {e})")
        return False

In [3]:
BASE_URL = os.environ.get("BASE_URL", "http://localhost:8000")
API_KEY = os.environ.get("API_KEY", "default_internal_key")
HEADERS = {"X-API-Key": API_KEY}

print(f"Testing API at: {BASE_URL}")
print(f"API Key: {API_KEY[:10]}...")

Testing API at: http://localhost:8000
API Key: default_in...


## 1. Health Check (No Auth Required)

In [4]:
response = requests.get(f"{BASE_URL}/healthz")
assert_status(response, 200, "Health Check")

# Verify health response structure
try:
    data = response.json()
    print(f"  Redis: {data.get('redis', 'N/A')}")
    print(f"  Browser Pool: {data.get('browser_pool', 'N/A')}")
except:
    pass

Health Check: ✓ PASS (200)
  Redis: connected
  Browser Pool: available (1 contexts)


## 2. Scraper (Stateless - Full Page)

In [25]:
payload = {
    "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8",
    "wait_until": "domcontentloaded"
}
response = requests.post(
    f"{BASE_URL}/scraper",
    headers=HEADERS,
    json=payload,
    timeout=60
)
scraper_ok = assert_status(response, 200, "Scraper")
if scraper_ok:
    assert_json_key(response, "id", "Scraper ID")
    assert_json_key(response, "url", "Scraper URL")
    assert_json_key(response, "content", "Scraper Content")

Scraper: ✓ PASS (200)
Scraper ID: ✓ PASS (key 'id' present)
Scraper URL: ✓ PASS (key 'url' present)
Scraper Content: ✓ PASS (key 'content' present)


In [29]:
html = json.loads(response.content.decode('utf-8'))['content']

## 3. Serper Search (Google via Playwright + Proxy)

In [46]:
payload = {
    "q": "What is artificial intelligence"
}
response = requests.post(
    f"{BASE_URL}/serper",
    headers=HEADERS,
    json=payload,
    timeout=120
)
serper_ok = assert_status(response, 200, "Serper Search")
if serper_ok:
    assert_json_key(response, "searchParameters", "Search Parameters")
    assert_json_key(response, "organic", "Organic Results")
    data = response.json()
    print(f"  Query: {data.get('searchParameters', {}).get('q', 'N/A')}")
    print(f"  Results: {len(data.get('organic', []))} found")

Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 0 found


In [48]:
for i in range(10):
    payload = {
        "q": "What is artificial intelligence"
    }
    response = requests.post(
        f"{BASE_URL}/serper",
        headers=HEADERS,
        json=payload,
        timeout=120
    )
    serper_ok = assert_status(response, 200, "Serper Search")
    if serper_ok:
        assert_json_key(response, "searchParameters", "Search Parameters")
        assert_json_key(response, "organic", "Organic Results")
        data = response.json()
        print(f"  Query: {data.get('searchParameters', {}).get('q', 'N/A')}")
        print(f"  Results: {len(data.get('organic', []))} found")

Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 10 found
Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 10 found
Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 0 found
Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 10 found
Serper Search: ✓ PASS (200)
Search Parameters: ✓ PASS (key 'searchParameters' present)
Organic Results: ✓ PASS (key 'organic' present)
  Query: What is artificial intelligence
  Results: 0 found
Serper Search: ✓ PASS 

In [32]:
response.content

b'{"searchParameters":{"q":"What is artificial intelligence","type":"search","engine":"google","num":10},"organic":[]}'

In [49]:
json.loads(response.content.decode('utf-8'))

{'searchParameters': {'q': 'What is artificial intelligence',
  'type': 'search',
  'engine': 'google',
  'num': 10},
 'organic': [{'title': 'IBM +5',
   'link': 'https://www.ibm.com/think/topics/artificial-intelligence',
   'snippet': '',
   'position': 1},
  {'title': 'Britannica +4',
   'link': 'https://www.britannica.com/technology/artificial-intelligence',
   'snippet': '',
   'position': 2},
  {'title': 'Artificial intelligence - WikipediaArtificial intelligence (AI) is the capability of computational s',
   'link': 'https://en.wikipedia.org/wiki/Artificial_intelligence',
   'snippet': 'ystems to perform tasks typically associated with human intellige...Wikipedia',
   'position': 3},
  {'title': 'What is Artificial intelligence (AI)? - Michigan Technological UniversityWhat is Artificial intellig',
   'link': 'https://www.mtu.edu/computing/ai/',
   'snippet': 'ence (AI)? Artificial intelligence (AI) encompasses the fields of computer and data science focused on ...Michigan Technol

## 4. Omni-Parse (AI Element Analysis)

In [ ]:
# AI element analysis - requires LLM server running
payload = {
    "base64_image": "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNk+M9QDwADhgGAWjR9awAAAABJRU5ErkJggg==",
    "prompt": "Find all clickable buttons"
}
response = requests.post(
    f"{BASE_URL}/omni-parse",
    headers=HEADERS,
    json=payload,
    timeout=30
)
assert_status(response, 200, "Omni-Parse")
if response.status_code == 200:
    assert_json_key(response, "elements", "Elements")

## 5. Jina Extract (AI Structured Extraction)

In [30]:
# AI structured extraction - requires LLM server
html_content = "<html><body><h1>Test Page</h1><p>Welcome to our site</p></body></html>"
payload = {
    "html": html_content,
    "extraction_schema": {
        "title": "string",
        "description": "string"
    }
}
response = requests.post(
    f"{BASE_URL}/jina-extract",
    headers=HEADERS,
    json=payload,
    timeout=30
)
jina_ok = assert_status(response, 200, "Jina Extract")
if jina_ok:
    assert_json_key(response, "extracted_data", "Extracted Data")

Jina Extract: ✗ FAIL (expected 200, got 404)
Status: 404
{
  "detail": "Not Found"
}


## 6. HTML to Markdown Conversion

In [37]:
payload = {
    "html": "<html><body><h1>Hello World</h1><p>This is a <strong>test</strong> page.</p></body></html>",
    "clean_html": True
}
response = requests.post(
    f"{BASE_URL}/api/v1/html-to-md",
    headers=HEADERS,
    json=payload,
    timeout=30
)
html2md_ok = assert_status(response, 200, "HTML to Markdown")
if html2md_ok:
    data = response.json()
    print(f"  Markdown: {data.get('markdown', 'N/A')[:200]}...")

HTML to Markdown: ✗ FAIL (expected 200, got 404)
Status: 404
{
  "detail": "Not Found"
}


## 7. Yandex Maps Extract

In [ ]:
payload = {
    "category": "restaurants",
    "center": {
        "lat": 59.934,
        "lng": 30.306
    },
    "radius": 1000
}
response = requests.post(
    f"{BASE_URL}/api/v1/yandex-maps/extract",
    headers=HEADERS,
    json=payload,
    timeout=120
)
yandex_ok = assert_status(response, 200, "Yandex Maps Extract")
if yandex_ok:
    assert_json_key(response, "businesses", "Businesses")
    data = response.json()
    print(f"  Businesses found: {len(data.get('businesses', []))}")

## 8. Website Enrichment

In [ ]:
payload = {
    "url": "https://example.com",
    "crawl_about": True,
    "crawl_services": False
}
response = requests.post(
    f"{BASE_URL}/api/v1/enrich",
    headers=HEADERS,
    json=payload,
    timeout=60
)
enrich_ok = assert_status(response, 200, "Website Enrichment")
if enrich_ok:
    assert_json_key(response, "title", "Title")
    assert_json_key(response, "content", "Content")

## 9. Research Agent - Start Task

In [ ]:
payload = {
    "query": "What is the capital of France?",
    "mode": "speed"
}
response = requests.post(
    f"{BASE_URL}/api/v1/research/run",
    headers=HEADERS,
    json=payload,
    timeout=30
)
assert_status(response, 202, "Research Task Started")

# Extract task_id for subsequent tests
task_id = None
if response.status_code == 202:
    try:
        task_id = response.json().get("task_id")
        print(f"  Task ID: {task_id}")
    except:
        pass

print(f"\nTask ID for later tests: {task_id}")

## 10. Research Agent - Get Status

In [ ]:
if task_id:
    response = requests.get(
        f"{BASE_URL}/api/v1/research/status/{task_id}",
        headers=HEADERS,
        timeout=30
    )
    assert_status(response, 200, "Research Status")
    if response.status_code == 200:
        data = response.json()
        print(f"  Status: {data.get('status', 'N/A')}")
        print(f"  Phase: {data.get('progress', {}).get('phase', 'N/A')}")
else:
    print("No task_id available - skipping status test")

## 11. Research Agent - Stream Progress (SSE)

In [ ]:
if task_id:
    response = requests.get(
        f"{BASE_URL}/api/v1/research/stream/{task_id}",
        headers=HEADERS,
        stream=True,
        timeout=30
    )
    if response.status_code == 200:
        content_type = response.headers.get('content-type', '')
        print(f"  Content-Type: {content_type}")
        assert 'text/event-stream' in content_type, "Expected SSE content type"
        print(f"  ✓ PASS - SSE stream received")
    else:
        print(f"  ✗ FAIL - Status: {response.status_code}")
else:
    print("No task_id available - skipping stream test")

## 12. Create Browser Session (Headless)

In [13]:
response = requests.post(
    f"{BASE_URL}/sessions",
    headers=HEADERS,
    json={"headless": False}
)
session_ok = assert_status(response, 200, "Create Session (Headless)")

session_id = None
if session_ok:
    session_id = response.json().get("session_id")
    print(f"  Session ID: {session_id}")

print(f"\nSession ID for later tests: {session_id}")

Create Session (Headless): ✓ PASS (200)
  Session ID: 296db574-57a2-4cb9-a0ba-be30c189aa3c

Session ID for later tests: 296db574-57a2-4cb9-a0ba-be30c189aa3c


## 13. Execute DSL Command on Session

In [21]:
if session_id:
    # Test goto command
    command = {
        "type": "goto",
        "params": {
            "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command goto: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: goto'}


In [22]:
if session_id:
    # Test goto command
    command = {
        "type": "scroll",
        "params": {
            "direction": "down",
            "amount": 400
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command scroll: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: scroll'}


In [21]:
if session_id:
    # Test goto command
    command = {
        "type": "goto",
        "params": {
            "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command goto: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: goto'}


In [21]:
if session_id:
    # Test goto command
    command = {
        "type": "goto",
        "params": {
            "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command goto: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: goto'}


In [21]:
if session_id:
    # Test goto command
    command = {
        "type": "goto",
        "params": {
            "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command goto: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: goto'}


In [21]:
if session_id:
    # Test goto command
    command = {
        "type": "scroll",
        "params": {
            "url": "https://ru.wikipedia.org/wiki/%D0%9A%D0%B0%D0%B7%D0%B0%D1%85%D1%81%D1%82%D0%B0%D0%BD_%D0%B2_%D1%81%D0%BE%D1%81%D1%82%D0%B0%D0%B2%D0%B5_%D0%A0%D0%BE%D1%81%D1%81%D0%B8%D0%B9%D1%81%D0%BA%D0%BE%D0%B9_%D0%B8%D0%BC%D0%BF%D0%B5%D1%80%D0%B8%D0%B8"
        }
    }
    response = requests.post(
        f"{BASE_URL}/sessions/{session_id}/command",
        headers=HEADERS,
        json=command,
        timeout=60
    )
    assert_status(response, 200, f"Execute Command {command['type']}")
    if response.status_code == 200:
        assert_json_key(response, "status", "Status")
        data = response.json()
        print(f"  Result Status: {data.get('status', 'N/A')}")
    print(f"\n{json.loads(response.content.decode('utf-8'))}")
else:
    print("No session_id available - skipping command test")

Execute Command goto: ✓ PASS (200)
Status: ✓ PASS (key 'status' present)
  Result Status: error

{'status': 'error', 'message': 'Unknown action: goto'}


In [18]:
response.content

b'{"status":"error","message":"Unknown action: goto"}'

## 14. Delete Session

In [32]:
if session_id:
    response = requests.delete(
        f"{BASE_URL}/sessions/{session_id}",
        headers=HEADERS
    )
    assert_status(response, 200, "Delete Session")
    print(f"  ✓ Session {session_id} deleted")
else:
    print("No session_id available - skipping delete test")

Delete Session: ✓ PASS (200)
  ✓ Session f58fd6bd-2ee2-4d63-82a8-85b2effc7ff7 deleted


## 15. Authentication Tests

In [ ]:
# Test without API key (should fail)
response = requests.post(
    f"{BASE_URL}/api/v1/research/run",
    json={"query": "test", "mode": "speed"}
)
assert_status(response, 403, "Without API Key")

# Test with invalid API key (should fail)
response = requests.post(
    f"{BASE_URL}/api/v1/research/run",
    headers={"X-API-Key": "invalid_key"},
    json={"query": "test", "mode": "speed"}
)
assert_status(response, 403, "With Invalid API Key")

# Test health check without auth (should succeed)
response = requests.get(f"{BASE_URL}/healthz")
assert_status(response, 200, "Health Check (No Auth)")

## Summary

### Test Results

| # | Endpoint | Status |
|---|----------|--------|
| 1 | `GET /healthz` | " + ("✓" if True else "✗") + " |
| 2 | `POST /scraper` | " + ("✓" if scraper_ok else "✗") + " |
| 3 | `POST /serper` | " + ("✓" if serper_ok else "✗") + " |
| 4 | `POST /omni-parse` | - |
| 5 | `POST /jina-extract` | - |
| 6 | `POST /api/v1/html-to-md` | " + ("✓" if html2md_ok else "✗") + " |
| 7 | `POST /api/v1/yandex-maps/extract` | " + ("✓" if yandex_ok else "✗") + " |
| 8 | `POST /api/v1/enrich` | " + ("✓" if enrich_ok else "✗") + " |
| 9 | `POST /api/v1/research/run` | - |
| 10 | `GET /api/v1/research/status/{task_id}` | - |
| 11 | `GET /api/v1/research/stream/{task_id}` | - |
| 12 | `POST /sessions` | " + ("✓" if session_ok else "✗") + " |
| 13 | `POST /sessions/{id}/command` | - |
| 14 | `DELETE /sessions/{id}` | - |
| 15 | Auth Tests | - |

### Notes
- Tests marked ✗ require specific infrastructure (proxies, LLM server, etc.)
- Research Agent tests require Taskiq worker to be running
- Yandex Maps requires residential proxies
- Omni-Parse and Jina Extract require LLM server

In [ ]:
print("\n" + "="*60)
print("API FULL TEST COMPLETE")
print("="*60)